# EPCLM-002-A0 — Persistent State Recall

One-click A0-v0.2 screening run. Enable Internet, select a GPU accelerator, and add the existing Kaggle Secret `GITHUB_TOKEN`, then use **Run All** once. It runs the bounded GPU training and runtime pilot, adapts the screening episode target to measured P95 time, and publishes a development-only report. The launcher reserves 600 seconds for publication inside a 5,400-second execution budget; the notebook enforces a 7,200-second hard limit. Checkpoints and incomplete evidence remain in Kaggle Output. This does not run the frozen formal matrix or produce a scientific decision.

In [ ]:
from pathlib import Path
import json, os, signal, subprocess, sys, time

BRANCH = 'codex/resolve-epclm001-remote-branch-conflicts'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-epclm-002-a0'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix':
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Hard two-hour limit reached. Kaggle Output remains under ' + str(REPO / 'results/epclm-002-a0'))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Kaggle Output remains under ' + str(REPO / 'results/epclm-002-a0'))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH, REPOSITORY, str(REPO)])

launcher = REPO / 'research/projects/cellular-language-models/experiments/epclm-002-a0-persistent-state-recall/scripts/kaggle_screening_launch.py'
publisher = REPO / 'scripts/research/publish_experiment_results.py'
PUBLISH_DIR = REPO / 'results/epclm-002-a0/publish'
identity_path = PUBLISH_DIR / 'execution-identity.json'
decision_path = PUBLISH_DIR / 'decision.json'
metrics_path = PUBLISH_DIR / 'metrics.json'
summary_path = PUBLISH_DIR / 'summary.md'

pending = None
if all(path.is_file() for path in (identity_path, decision_path, metrics_path, summary_path)):
    saved_identity = json.loads(identity_path.read_text(encoding='utf-8'))
    saved_decision = json.loads(decision_path.read_text(encoding='utf-8'))
    source_sha = str(saved_identity.get('source_commit', ''))
    saved_run_id = str(saved_identity.get('run_id', ''))
    valid_sha = len(source_sha) == 40 and all(ch in '0123456789abcdef' for ch in source_sha.lower())
    if (
        saved_identity.get('plan_id') == 'A0-v0.2-screening'
        and saved_decision.get('plan_id') == 'A0-v0.2-screening'
        and saved_decision.get('source_commit') == source_sha
        and valid_sha
        and saved_run_id == source_sha[:12] + '-screening-v0.2'
    ):
        prior_run = REPO / 'results/epclm-002-a0/runs' / saved_run_id
        if not (prior_run / 'publication-receipt.json').is_file():
            pending = (saved_identity, prior_run)

if pending is not None:
    saved_identity, prior_run = pending
    result_branch = 'kaggle/epclm-002-a0-screen-v02-' + saved_identity['source_commit'][:12]
    print('Found completed A0-v0.2 output awaiting publication. Retrying the existing publisher without rerunning training or evaluation.')
    run_bounded([
        sys.executable, str(publisher), 'epclm-002-a0', '--push',
        '--branch', result_branch, '--base-ref', 'origin/main'
    ], cwd=REPO, timeout_seconds=600)
    remote = subprocess.run(
        ['git', '-C', str(REPO), 'ls-remote', '--heads', 'origin', 'refs/heads/' + result_branch],
        check=True, text=True, capture_output=True
    ).stdout.strip().split()
    if not remote:
        raise RuntimeError('Publisher completed but the result branch could not be verified.')
    (prior_run / 'publication-receipt.json').write_text(
        json.dumps({
            'status': 'VERIFIED',
            'branch': result_branch,
            'remote_sha': remote[0],
            'source_commit': saved_identity['source_commit'],
            'verified_by_publisher': True
        }, indent=2, sort_keys=True) + '\n',
        encoding='utf-8'
    )
    print('Saved v0.2 results were published and the remote commit SHA was verified.')
else:
    run_bounded([sys.executable, str(launcher), '--max-runtime-seconds', '6000'], cwd=REPO)
    print('Run finished. The v0.2 screening report was returned to GitHub; pilot timings, checkpoints, per-episode evidence, and incomplete cells remain in Kaggle Output.')